In [ ]:
# Imports and Setup
import subprocess
import json

# Define your models and input prompts
models = [
    "mistral:7b", "deepseek-r1:8b", "llama3.1:8b", "gemma:7b"
]

prompts = [
    {
        "ticker": "MSFT",
        "action": "BUY",
        "pe": 32.1,
        "rsi": 58,
        "beta": 1.1
    },
    {
        "ticker": "NVDA",
        "action": "HOLD",
        "pe": 75.6,
        "rsi": 69,
        "beta": 1.8
    },
    {
        "ticker": "AMZN",
        "action": "SELL",
        "pe": 92.4,
        "rsi": 73,
        "beta": 1.4
    }
]


In [ ]:
# Prompt Template Function
def build_prompt(data):
    return f"""You are a financial assistant. Based on the following metrics:

Ticker: {data['ticker']}
Action: {data['action']}
P/E Ratio: {data['pe']}
RSI: {data['rsi']}
Beta: {data['beta']}

Generate a short, clear explanation of why this action was recommended, using only the signals provided.

Your explanation will be evaluated based on:
1. Factual alignment – Does it correctly reference and reflect the input signals?
2. Clarity – Is it readable and jargon-free for a non-expert?
3. Educational value – Does it help the user understand financial reasoning?

Write in a tone that is simple, supportive, and instructive.
"""


In [ ]:
# Run Inference (Ollama CLI)
def query_ollama(model: str, prompt: str) -> str:
    try:
        result = subprocess.run(
            ["ollama", "run", model],
            input=prompt,
            text=True,
            capture_output=True,
            timeout=60
        )
        return result.stdout.strip()
    except Exception as e:
        return f"Error: {str(e)}"


In [ ]:
results = []

for model in models:
    for data in prompts:
        prompt = build_prompt(data)
        print(f"Running model: {model} on {data['ticker']}")
        response = query_ollama(model, prompt)
        results.append({
            "model": model,
            "ticker": data["ticker"],
            "action": data["action"],
            "pe":data["pe"],
            "rsi":data["rsi"],
            "beta":data["beta"],
            "response": response
        })


In [ ]:
import pandas as pd
df = pd.DataFrame(results)
df.head()


In [ ]:
print("Ticker: ", df.ticker[0])
print("Action: ", df.action[0])
print("P/E: ", df.pe[0])
print("RSI: ", df.rsi[0])
print("beta: ", df.beta[0])

In [ ]:
print("Model: ", df.model[0])
print("Response: ", df.response[0])

In [ ]:
print("Model: ", df.model[3])
print("Response: ", df.response[3])

In [ ]:
print("Model: ", df.model[6])
print("Response: ", df.response[6])

In [ ]:
print("Model: ", df.model[9])
print("Response: ", df.response[9])

In [ ]:
# ---

In [ ]:
import pandas as pd
import random
import subprocess
import re
import ast
import matplotlib.pyplot as plt

models = df["model"].unique().tolist()
tickers = df["ticker"].unique().tolist()
grouped = {ticker: df[df["ticker"] == ticker].to_dict("records") for ticker in tickers}

def build_prompt(ticker, action, pe, rsi, beta, anonymised):
    prompt = f"You are evaluating explanations for:\n" \
             f"Ticker: {ticker}, Action: {action}, P/E: {pe}, RSI: {rsi}, Beta: {beta}\n\n" \
             f"Explanations:\n"
    for r in anonymised:
        prompt += f"{r['id']}: {r['response']}\n"
    prompt += (
        "\nEvaluate each explanation on 1–5 (with 5 being excellent!) for:\n"
        "1. Factual alignment with input signals\n"
        "2. Clarity for non-experts\n"
        "3. Educational value\n\n"
        "Return only a JSON object like:\n"
        "{ \"Resp 1\": [5, 4, 5], \"Resp 2\": [4, 3, 5], ... }"
    )
    return prompt

def safe_parse_json(text):
    try:
        match = re.search(r"\{.*?\}", text, re.DOTALL)
        if not match:
            return None
        return ast.literal_eval(match.group())
    except Exception:
        return None

all_scores = []

for ticker in tickers:
    rows = grouped[ticker]
    for judge in models:
        for run in range(3):  # Repeat evaluation 3 times
            shuffled = random.sample(rows, len(rows))
            anonymised = [{"id": f"Resp {i+1}", "response": r["response"]} for i, r in enumerate(shuffled)]
            id_to_model = {f"Resp {i+1}": r["model"] for i, r in enumerate(shuffled)}

            prompt = build_prompt(
                ticker=ticker,
                action=rows[0]["action"],
                pe=rows[0]["pe"],
                rsi=rows[0]["rsi"],
                beta=rows[0]["beta"],
                anonymised=anonymised
            )

            result = subprocess.run(
                ["ollama", "run", judge],
                input=prompt,
                text=True,
                capture_output=True,
                timeout=90
            )

            output = result.stdout.strip()
            parsed = safe_parse_json(output)

            if parsed:
                for rid, scores in parsed.items():
                    all_scores.append({
                        "evaluator": judge,
                        "model": id_to_model.get(rid),
                        "ticker": ticker,
                        "response_id": rid,
                        "run": run + 1,
                        "score_factual": scores[0],
                        "score_clarity": scores[1],
                        "score_learning": scores[2]
                    })
            else:
                print(f"⚠️ Failed to parse response from {judge} on {ticker} (run {run+1})")
                print(output)


In [ ]:
# === Score aggregation and weighted ranking ===
df_scores = pd.DataFrame(all_scores)

# Mean per model
agg = (
    df_scores.groupby("model")[["score_factual", "score_clarity", "score_learning"]]
    .mean()
    .round(2)
)

# Apply custom weights (fluency weighted higher)
w_factual = 0.3
w_clarity = 0.4
w_learning = 0.3

agg["mean_weighted"] = (
    agg["score_factual"] * w_factual +
    agg["score_clarity"] * w_clarity +
    agg["score_learning"] * w_learning
).round(2)

# Sort by weighted score
agg = agg.sort_values(by="mean_weighted", ascending=False)

# Display result
print("\n📊 Weighted Model Rankings (Fluency Emphasised):")
print(agg[["score_factual", "score_clarity", "score_learning", "mean_weighted"]])

# Optional: bar chart
agg[["score_factual", "score_clarity", "score_learning"]].plot(
    kind="bar", figsize=(10, 6), title="Model Evaluation by Criterion"
)
plt.ylabel("Average Score (1–5)")
plt.ylim(0, 5)
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.tight_layout()
plt.show()